In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [6]:
x = torch.tensor([[3,1,4,2,5]])
a = torch.tensor([[0,1,2,3,4]])
n_embd = 8
head_size = 2
block_size = 5

In [3]:
from zero_to_hero.part_04_gpt.attention import MultiHeadAttention

In [4]:
mha = MultiHeadAttention(n_embd, 4, head_size, block_size, 0.0)

In [7]:
tet = nn.Embedding(6, n_embd)
pet = nn.Embedding(block_size, n_embd)
inp = tet(x) + pet(a)

In [9]:
y = mha(inp)

In [10]:
y

tensor([[[-1.2516,  0.4814,  0.0318,  0.1563,  0.2176, -0.5703,  0.9541,
           0.3104],
         [-0.7114,  0.1748, -0.0265,  0.1787,  0.3914, -0.4442,  0.4712,
           0.2403],
         [-0.3467,  0.0027, -0.1371,  0.1906,  0.6302, -0.3320,  0.0057,
          -0.0508],
         [-0.6135,  0.0025, -0.0487,  0.1886,  0.7433, -0.5221,  0.2198,
           0.0748],
         [-0.0462,  0.0630, -0.3974,  0.0095,  0.9206,  0.2823, -0.2821,
          -0.2058]]], grad_fn=<ViewBackward0>)

In [72]:
class MultiHeadCombined(nn.Module):
    """ one head of self-attention """

    def __init__(self, n_embd: int, num_heads: int, head_size: int, block_size: int, dropout: float):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(n_embd, n_embd, bias=False)
        self.query = nn.Linear(n_embd, n_embd, bias=False)
        self.value = nn.Linear(n_embd, n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))
        self.dropout = nn.Dropout(dropout)
        self.num_heads = num_heads
        self.head_size = head_size

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x).view(B, T, self.num_heads, self.head_size).transpose(2,1)
        q = self.query(x).view(B, T, self.num_heads, self.head_size).transpose(2,1)
        # compute attention scores -  "affinities"
        wei = q @ k.transpose(-2,-1) * (self.head_size ** -0.5)
        wei = wei.masked_fill(self.tril[:, :, :T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        # weighted aggregation of the values
        v = self.value(x).view(B, T, self.num_heads, self.head_size).transpose(2,1)
        # perform aggregation
        out = wei @ v 
        # reshape
        out = out.transpose(2,1).contiguous().view(B,T,C)

        # projection and dropout
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [73]:
mhac = MultiHeadCombined(n_embd, 4, head_size, block_size, 0.0)

In [74]:
yc = mhac(inp)

In [38]:
B, T, nh, hs = 1, 3, 2, 2
a = torch.arange(B*T*nh*hs).float().view(B, T, nh*hs)   # (1,3,4)
print(a[0])                                    # row t is [4t, 4t+1, 4t+2, 4t+3]
print(a.view(B, nh, T, hs)[0, 0])            # which timesteps ended up in head 0?

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])
tensor([[0., 1.],
        [2., 3.],
        [4., 5.]])


In [29]:
a

tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]])

In [39]:
a.view(B, nh, T, hs).shape

torch.Size([1, 2, 3, 2])

In [48]:
a.view(B, nh, T, hs).transpose(2,1)[0,:,0,:]

tensor([[0., 1.],
        [2., 3.],
        [4., 5.]])

In [50]:
print(a.view(B, T, nh, hs).transpose(2,1)[0, 0])  

tensor([[0., 1.],
        [4., 5.],
        [8., 9.]])


In [63]:
wei.shape

torch.Size([1, 4, 5, 5])

In [64]:
vc.shape

torch.Size([1, 4, 5, 2])

In [65]:
yc.shape

torch.Size([1, 4, 5, 2])

In [71]:
yc.transpose(2,1).contiguous().view(1, block_size, n_embd)

tensor([[[-1.3891,  0.6656,  0.4971,  0.9864,  1.4679, -1.0671,  1.2030,
           0.9849],
         [-0.7090,  0.0709, -0.1095, -0.2351,  0.4578, -0.0821,  0.5232,
           0.2718],
         [-0.4701, -0.4696, -0.1159, -0.5201,  0.4878, -0.0962,  0.1997,
           0.0734],
         [-0.2439, -0.1189, -0.2274, -0.5193,  0.5674,  0.1833,  0.0244,
           0.0979],
         [-0.4818, -0.1096, -0.2884, -0.2358,  0.5974, -0.1009, -0.0954,
          -0.1423]]], grad_fn=<ViewBackward0>)

In [78]:
y.shape

torch.Size([1, 5, 8])

In [77]:
yc.shape

torch.Size([1, 5, 8])

In [80]:
# Reproducibility
torch.manual_seed(42)

# Example dimensions
B = 2
T = 4
n_embd = 32
num_heads = 4
head_size = n_embd // num_heads
block_size = 8
dropout = 0.0

# Create models
mha = MultiHeadAttention(
    n_embd=n_embd,
    num_heads=num_heads,
    head_size=head_size,
    block_size=block_size,
    dropout=dropout,
)

combined = MultiHeadCombined(
    n_embd=n_embd,
    num_heads=num_heads,
    head_size=head_size,
    block_size=block_size,
    dropout=dropout,
)

# Copy weights
with torch.no_grad():
    combined.key.weight.copy_(
        torch.cat([h.key.weight for h in mha.heads], dim=0)
    )
    combined.query.weight.copy_(
        torch.cat([h.query.weight for h in mha.heads], dim=0)
    )
    combined.value.weight.copy_(
        torch.cat([h.value.weight for h in mha.heads], dim=0)
    )

    combined.proj.weight.copy_(mha.proj.weight)
    combined.proj.bias.copy_(mha.proj.bias)

# Disable dropout
mha.eval()
combined.eval()

# Same input
x = torch.randn(B, T, n_embd)

# Forward pass
with torch.no_grad():
    y1 = mha(x)
    y2 = combined(x)

# Compare
print("Outputs equal:", torch.allclose(y1, y2, atol=1e-6))
print("Max difference:", (y1 - y2).abs().max().item())

Outputs equal: True
Max difference: 1.1920928955078125e-07


In [82]:
sum(p.numel() for p in mha.parameters()) == sum(p.numel() for p in combined.parameters())

True